# Breast Cancer Segmentation — Algorithm Benchmarking

This notebook benchmarks multiple segmentation algorithms against ground-truth masks using:
- ROC Curve & AUC (manual + sklearn)
- Jaccard Index & Dice Coefficient
- Hausdorff Distance (boundary accuracy)


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
from __future__ import annotations

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from scipy.spatial import cKDTree
from sklearn.metrics import roc_curve, auc

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
ROOT          = Path("images")
GT_PATH       = ROOT / "ground_truth"
ALGO_PATH     = ROOT / "algorithms"

BIN_THRESH    = 127     # pixel value threshold for binarisation
NORM_FACTOR   = 255.0   # converts uint8 → [0, 1] probability scores

## 1 · Image Loading

In [ ]:
def read_grayscale_images(folder: Path) -> Tuple[List[np.ndarray], List[str]]:
    """
    Load every readable grayscale image from *folder* in sorted order.

    Returns
    -------
    pixel_arrays : list of 2-D uint8 arrays
    names        : corresponding filenames
    """
    pixel_arrays, names = [], []
    for path in sorted(folder.iterdir()):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            pixel_arrays.append(img)
            names.append(path.name)
    return pixel_arrays, names

## 2 · Visualisation

In [ ]:
def show_ground_truth(masks: List[np.ndarray], names: List[str]) -> None:
    """Display all ground-truth masks side by side."""
    n = len(masks)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    fig.suptitle("Ground-Truth Masks", fontsize=16, fontweight="bold")

    axes = [axes] if n == 1 else axes
    for ax, mask, name in zip(axes, masks, names):
        ax.imshow(mask, cmap="gray")
        ax.set_title(name, fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def show_prediction_grid(
    predictions: Dict[str, List[np.ndarray]],
    algo_names: List[str],
    img_names: List[str],
) -> None:
    """
    Render a (algorithms × images) grid of segmentation predictions.

    Parameters
    ----------
    predictions : algo_name → list of prediction arrays
    algo_names  : ordered list of algorithm identifiers
    img_names   : ordered list of image filenames (column headers)
    """
    n_algos, n_imgs = len(algo_names), len(img_names)
    fig, axes = plt.subplots(
        n_algos, n_imgs, figsize=(4 * n_imgs, 3 * n_algos)
    )
    fig.suptitle("Algorithm Predictions", fontsize=20, fontweight="bold", y=1)

    # Normalise axes to a 2-D array regardless of dimensions
    axes = np.atleast_2d(axes)
    if n_algos == 1:
        axes = axes[np.newaxis, :] if axes.ndim == 1 else axes
    if n_imgs == 1:
        axes = axes[:, np.newaxis]

    for row, algo in enumerate(algo_names):
        for col in range(n_imgs):
            ax = axes[row, col]
            ax.imshow(predictions[algo][col], cmap="gray")
            if row == 0:
                ax.set_title(img_names[col], fontsize=12, fontweight="bold")
            if col == 0:
                ax.set_ylabel(algo, fontsize=13, fontweight="bold")
            ax.set_xticks([])
            ax.set_yticks([])

    plt.tight_layout()
    plt.show()


def display_all_images() -> None:
    """Convenience wrapper: load and display GT + algorithm outputs."""
    gt_masks, gt_names = read_grayscale_images(GT_PATH)
    algo_names = sorted(
        d.name for d in ALGO_PATH.iterdir() if d.is_dir()
    )
    preds = {
        algo: read_grayscale_images(ALGO_PATH / algo)[0]
        for algo in algo_names
    }

    print("▶ Ground-Truth masks")
    show_ground_truth(gt_masks, gt_names)

    print("▶ Algorithm prediction grid")
    show_prediction_grid(preds, algo_names, gt_names)


display_all_images()

## 3 · ROC, Jaccard & Dice

In [ ]:
def to_labels_and_scores(
    gt_stack: List[np.ndarray],
    pred_stack: List[np.ndarray],
) -> Tuple[np.ndarray, np.ndarray]:
    """Flatten image stacks into 1-D binary labels + soft probability scores."""
    gt_flat   = np.concatenate([m.ravel() for m in gt_stack])
    pred_flat = np.concatenate([p.ravel() for p in pred_stack])

    labels = (gt_flat > BIN_THRESH).astype(np.int32)
    scores = pred_flat / NORM_FACTOR
    return labels, scores


# ── Manual ROC implementation ─────────────────────────────────────────────────

def _roc_from_scratch(
    labels: np.ndarray, scores: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute ROC curve without any library.

    Returns
    -------
    fpr, tpr, thresholds  (all sorted by descending threshold)
    """
    unique_scores = np.unique(scores)[::-1]
    thresholds = np.concatenate(([np.inf], unique_scores))

    n_pos = labels.sum()
    n_neg = len(labels) - n_pos

    fpr_vals, tpr_vals = [], []
    for t in thresholds:
        predicted = (scores >= t).astype(np.int32)
        tp = int(((predicted == 1) & (labels == 1)).sum())
        fp = int(((predicted == 1) & (labels == 0)).sum())
        tpr_vals.append(tp / n_pos if n_pos else 0.0)
        fpr_vals.append(fp / n_neg if n_neg else 0.0)

    return np.asarray(fpr_vals), np.asarray(tpr_vals), thresholds


def _trapezoid_auc(fpr: np.ndarray, tpr: np.ndarray) -> float:
    """Trapezoidal-rule AUC over a manually computed ROC curve."""
    total = 0.0
    for i in range(1, len(fpr)):
        dx = fpr[i] - fpr[i - 1]
        total += dx * (tpr[i] + tpr[i - 1]) / 2.0
    return total


# ── Overlap metrics ───────────────────────────────────────────────────────────

def overlap_metrics(
    labels: np.ndarray, scores: np.ndarray, threshold: float
) -> Tuple[float, float]:
    """
    Jaccard index and Dice coefficient at a fixed *threshold*.

    Returns (jaccard, dice).
    """
    predicted = (scores >= threshold).astype(np.int32)

    if labels.sum() == 0 and predicted.sum() == 0:
        return 1.0, 1.0

    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())

    denom = tp + fp + fn
    jaccard = tp / denom if denom else 0.0
    dice    = (2 * jaccard) / (1 + jaccard)
    return jaccard, dice


def best_jaccard_threshold(
    labels: np.ndarray,
    scores: np.ndarray,
    thresholds: np.ndarray,
) -> Tuple[int, float, float, float]:
    """
    Sweep *thresholds* and return the index + value that maximises Jaccard.

    Returns
    -------
    best_idx, best_threshold, best_jaccard, corresponding_dice
    """
    jaccard_scores = [
        overlap_metrics(labels, scores, t)[0] for t in thresholds
    ]
    idx = int(np.argmax(jaccard_scores))
    j   = jaccard_scores[idx]
    d   = (2 * j) / (1 + j)
    return idx, thresholds[idx], j, d


def best_roc_threshold(
    fpr: np.ndarray, tpr: np.ndarray, thresholds: np.ndarray
) -> Tuple[int, float]:
    """
    Optimal operating point on the ROC curve via Euclidean distance to (0, 1).

    Returns (index_in_array, threshold_value).
    """
    dist = np.hypot(fpr, tpr - 1.0)
    idx  = int(np.argmin(dist))
    return idx, thresholds[idx]


# ── Combined ROC computation ──────────────────────────────────────────────────

def compute_roc_bundle(
    labels: np.ndarray, scores: np.ndarray
) -> dict:
    """
    Compute ROC curves and AUC via both manual and sklearn pipelines.

    Returns a dict with keys:
        fpr_m, tpr_m, auc_m, thresholds_m,
        fpr_sk, tpr_sk, auc_sk
    """
    fpr_m, tpr_m, thresh_m = _roc_from_scratch(labels, scores)
    auc_m = _trapezoid_auc(fpr_m, tpr_m)

    fpr_sk, tpr_sk, _ = roc_curve(labels, scores, drop_intermediate=False)
    auc_sk = auc(fpr_sk, tpr_sk)

    return dict(
        fpr_m=fpr_m, tpr_m=tpr_m, auc_m=auc_m, thresholds_m=thresh_m,
        fpr_sk=fpr_sk, tpr_sk=tpr_sk, auc_sk=auc_sk,
    )

## 4 · Hausdorff Distance

In [ ]:
def _extract_boundary(binary_mask: np.ndarray) -> np.ndarray:
    """
    Morphological boundary extraction via erosion subtraction.

    Returns an (N, 2) array of boundary pixel coordinates.
    """
    kernel   = np.ones((3, 3), dtype=np.uint8)
    eroded   = cv2.erode(binary_mask, kernel, iterations=1)
    boundary = binary_mask - eroded
    return np.argwhere(boundary > 0)


def _one_sided_hausdorff(
    source_pts: np.ndarray, target_pts: np.ndarray
) -> float:
    """max over source of min-distances to target (directed Hausdorff)."""
    tree = cKDTree(target_pts)
    dists, _ = tree.query(source_pts)
    return float(dists.max())


def hausdorff_distance(
    gt_img: np.ndarray,
    pred_img: np.ndarray,
    threshold: float,
) -> float:
    """
    Symmetric Hausdorff distance between boundary pixels of GT and prediction.

    Parameters
    ----------
    gt_img    : ground-truth grayscale image (uint8)
    pred_img  : probability-score image in [0, 1]
    threshold : binarisation cut-off for *pred_img*

    Returns
    -------
    Hausdorff distance in pixels, or 0.0 / inf for edge cases.
    """
    gt_bin   = (gt_img   >  0        ).astype(np.uint8)
    pred_bin = (pred_img >= threshold ).astype(np.uint8)

    gt_pts   = _extract_boundary(gt_bin)
    pred_pts = _extract_boundary(pred_bin)

    if len(gt_pts) == 0 and len(pred_pts) == 0:
        return 0.0
    if len(gt_pts) == 0 or len(pred_pts) == 0:
        return np.inf

    return max(
        _one_sided_hausdorff(gt_pts, pred_pts),
        _one_sided_hausdorff(pred_pts, gt_pts),
    )

## 5 · Metrics Reporting

In [ ]:
def _hausdorff_summary(
    gt_imgs: List[np.ndarray],
    pred_imgs: List[np.ndarray],
    threshold: float,
) -> str:
    """Compute per-image Hausdorff distances and return a formatted string."""
    hd_vals = [
        hausdorff_distance(gt, pred / NORM_FACTOR, threshold)
        for gt, pred in zip(gt_imgs, pred_imgs)
    ]

    if len(hd_vals) == 1:
        return f"{hd_vals[0]:.4f}"

    mn, mx, avg = np.min(hd_vals), np.max(hd_vals), np.mean(hd_vals)
    return f"{avg:.4f}  [{mn:.4f} – {mx:.4f}]"


def evaluate_algorithm(
    algo_name: str,
    labels: np.ndarray,
    scores: np.ndarray,
    gt_imgs: List[np.ndarray],
    pred_imgs: List[np.ndarray],
) -> dict:
    """
    Run the full evaluation pipeline for one algorithm.

    Prints one metrics row and returns a result dict for plotting.
    """
    roc = compute_roc_bundle(labels, scores)

    roc_idx, roc_thresh = best_roc_threshold(
        roc["fpr_m"], roc["tpr_m"], roc["thresholds_m"]
    )
    j_roc, d_roc = overlap_metrics(labels, scores, roc_thresh)

    jac_idx, jac_thresh, j_jac, d_jac = best_jaccard_threshold(
        labels, scores, roc["thresholds_m"]
    )

    hd_roc = _hausdorff_summary(gt_imgs, pred_imgs, roc_thresh)
    hd_jac = _hausdorff_summary(gt_imgs, pred_imgs, jac_thresh)

    W = dict(a=15, b=10, c=35, d=15, e=18, f=35)
    print(
        f"{algo_name:<{W['a']}} | {roc['auc_m']:<{W['b']}.4f} "
        f"| {roc_thresh:<{W['c']}.4f} | {j_roc:<{W['d']}.4f} "
        f"| {d_roc:<{W['e']}.4f} | {hd_roc:<{W['f']}} "
        f"| {jac_thresh:<{W['c']}.4f} | {j_jac:<{W['d']}.4f} "
        f"| {d_jac:<{W['e']}.4f} | {hd_jac:<{W['f']}}"
    )

    return dict(
        fpr_m=roc["fpr_m"], tpr_m=roc["tpr_m"], auc_m=roc["auc_m"],
        fpr_sk=roc["fpr_sk"], tpr_sk=roc["tpr_sk"], auc_sk=roc["auc_sk"],
        best_threshold_idx_roc=roc_idx,
        best_threshold_idx_jaccard=jac_idx,
    )


# ── ROC Comparison Plot ───────────────────────────────────────────────────────

def plot_roc_comparison(title: str, results: Dict[str, dict]) -> None:
    """
    Side-by-side ROC plots (manual vs sklearn) with optimal-threshold markers.
    """
    fig, (ax_m, ax_sk) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for algo, r in results.items():
        label_m  = f"{algo}  (AUC = {r['auc_m']:.4f})"
        label_sk = f"{algo}  (AUC = {r['auc_sk']:.4f})"

        ax_m.plot(r["fpr_m"],  r["tpr_m"],  lw=2, label=label_m)
        ax_sk.plot(r["fpr_sk"], r["tpr_sk"], lw=2, label=label_sk)

        for ax, fpr_key, tpr_key in [
            (ax_m,  "fpr_m",  "tpr_m"),
            (ax_sk, "fpr_sk", "tpr_sk"),
        ]:
            fpr, tpr = r[fpr_key], r[tpr_key]
            ax.plot(
                fpr[r["best_threshold_idx_roc"]],
                tpr[r["best_threshold_idx_roc"]],
                "o", color="red", ms=8,
            )
            ax.plot(
                fpr[r["best_threshold_idx_jaccard"]],
                tpr[r["best_threshold_idx_jaccard"]],
                "o", color="green", ms=8,
            )

    for ax, subtitle in [(ax_m, "Manual ROC"), (ax_sk, "Sklearn ROC")]:
        ax.plot([0, 1], [0, 1], ":", color="gray", label="Random baseline")
        ax.plot([], [], "o", color="red",   ms=8, linestyle="None", label="Best threshold (ROC)")
        ax.plot([], [], "o", color="green", ms=8, linestyle="None", label="Best threshold (Jaccard)")
        ax.set(xlim=[0, 1], ylim=[0, 1.05],
               xlabel="False Positive Rate", ylabel="True Positive Rate",
               title=subtitle)
        ax.legend(loc="lower right")
        ax.grid(True)

    plt.tight_layout()
    plt.show()

## 6 · Main Evaluation Pipeline

In [ ]:
# ── Column header helpers ─────────────────────────────────────────────────────
_COLS = [
    ("Algorithm",               15),
    ("AUC",                     10),
    ("Thresh (ROC)",            35),
    ("Jaccard",                 15),
    ("Dice",                    18),
    ("Hausdorff (px)",          35),
    ("Thresh (Jaccard)",        35),
    ("Jaccard",                 15),
    ("Dice",                    18),
    ("Hausdorff (px)",          35),
]

HEADER  = " | ".join(f"{name:<{w}}" for name, w in _COLS)
DIVIDER = "─" * len(HEADER)


def run_full_evaluation(gt_dir: Path, algo_dir: Path) -> None:
    """
    Entry point: evaluate all algorithms per-image and overall,
    then produce comparison plots.
    """
    gt_imgs, gt_names = read_grayscale_images(gt_dir)
    algo_names = sorted(d.name for d in algo_dir.iterdir() if d.is_dir())
    pred_bank  = {
        algo: read_grayscale_images(algo_dir / algo)[0]
        for algo in algo_names
    }

    # ── Per-image evaluation ──────────────────────────────────────────────────
    print(f"\n{'═' * 60}")
    print("  PER-IMAGE EVALUATION")
    print(f"{'═' * 60}")

    for img_idx, (gt, gt_name) in enumerate(zip(gt_imgs, gt_names)):
        print(f"\n  Image: {gt_name}")
        print(HEADER)
        print(DIVIDER)

        img_results = {}
        for algo in algo_names:
            pred = pred_bank[algo][img_idx]
            labels, scores = to_labels_and_scores([gt], [pred])
            img_results[algo] = evaluate_algorithm(
                algo, labels, scores, [gt], [pred]
            )

        plot_roc_comparison(f"Per-image ROC — {gt_name}", img_results)

    # ── Aggregate evaluation ──────────────────────────────────────────────────
    print(f"\n{'═' * 60}")
    print("  AGGREGATE EVALUATION (all images combined)")
    print(f"{'═' * 60}")
    print(HEADER)
    print(DIVIDER)

    overall_results = {}
    for algo in algo_names:
        all_preds         = pred_bank[algo]
        labels_all, scores_all = to_labels_and_scores(gt_imgs, all_preds)
        overall_results[algo] = evaluate_algorithm(
            algo, labels_all, scores_all, gt_imgs, all_preds
        )

    plot_roc_comparison("Aggregate ROC — All Algorithms", overall_results)


run_full_evaluation(GT_PATH, ALGO_PATH)

## 7 · Discussion & Conclusion

### Metrics Considered

**AUC (Area Under the ROC Curve)**  
AUC aggregates classification performance across all possible decision thresholds. While a useful screening metric, it was found to be *insufficient* here: the algorithm with the highest AUC did not achieve the best spatial overlap with the ground truth.

**Euclidean Distance to Ideal ROC Point**  
Using the threshold closest to the ideal operating point (0, 1) was investigated as a threshold selection strategy. In practice this heuristic was inconsistent—it did not reliably identify thresholds that maximised segmentation accuracy.

**Jaccard Index (Intersection over Union)**  
Jaccard directly measures spatial overlap between predicted and ground-truth segmentation masks. It was used both to *select the best algorithm* and to *optimise the decision threshold*. The algorithm with the highest Jaccard score was retained as the winner.

**Hausdorff Distance**  
Boundary precision was assessed via Hausdorff Distance:
- One algorithm achieved a seemingly better *average* Hausdorff value, but only because it always produced masks—even when they were grossly inaccurate.
- The selected algorithm incurred an *infinite* penalty on a single image (blank-mask failure), yet demonstrated highly precise boundary delineation across all other test cases.

---

### Conclusion

**Algorithm 3** was identified as the optimal segmentation algorithm based on:

1. **Highest Jaccard Index** — best spatial overlap with ground truth across the dataset.
2. **Superior Boundary Precision** — tightest boundary adherence on successful predictions.

The isolated infinite Hausdorff penalty (blank-mask edge case) does not reflect systemic failure and can be mitigated through upstream preprocessing (e.g., quality-checking inputs or applying a fallback detector). Discarding a high-performing algorithm in favour of a consistently mediocre one solely on this single-image artefact would be counterproductive.

> **Final recommendation:** Algorithm 3 at its Jaccard-optimal threshold, with targeted preprocessing applied to the identified failure cases.
